In [ ]:
### RAG Pipeline- DATA INGESTION TO VECTOR DB PIPELINE

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, DirectoryLoader, TextLoader

dir = DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt",
    loader_cls = TextLoader,
    show_progress = False
)

txt_load = dir.load()


C:\Users\Atharva\AppData\Local\Temp\ipykernel_10928\4048237288.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, DirectoryLoader


In [1]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader
def process_all_files(text_dir):
    all_documents = []
    txt_dir = Path(text_dir)
    txt_files = list(txt_dir.glob("**/*.txt"))
    print(F"found{len(txt_files)} text files to process")
    for txt_file in txt_files:
        print(f"\n processing: {txt_file.name}")
        try:
            loader = TextLoader(str(txt_file), encoding="utf-8")
            documents = loader.load()
            for doc in documents:
                doc.metadata['source_file'] = txt_file.name
                doc.metadata['file_type'] = "text file"
            all_documents.extend(documents)
            print(f"loaded {len(documents)} pages")
        except Exception as e:
            print(f"error :{e}")
    print(f"\n total document loaded :{len(all_documents)}")
    return all_documents

all_txt_documents = process_all_files("../data")

C:\Users\Atharva\AppData\Local\Temp\ipykernel_25868\1996310493.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
d:\projects\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


found7 text files to process

 processing: testing.txt
error :Error loading ..\data\pdf\testing.txt

 processing: Credit_card_bill_payment.txt
loaded 1 pages

 processing: HDFC_MITC.txt
loaded 1 pages

 processing: Pay_with_credit_card.txt
loaded 1 pages

 processing: RBI_Compliance_Framework.txt
loaded 1 pages

 processing: SBI_MITC.txt
loaded 1 pages

 processing: TERMS_CONDITIONS.txt
loaded 1 pages

 total document loaded :6


In [8]:
###SPLITTING INTO CHUNKS

from langchain_text_splitters import RecursiveCharacterTextSplitter
def split_document(documents,chunk_size=1000,chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print("\n example chunks:")
        print(f"content: {split_docs[0].page_content[:200]}")
        print(f"metadata : {split_docs[0].metadata}")
    return split_docs

In [9]:
chunks = split_document(all_txt_documents)
print(chunks)

split 6 documents into 34 chunks

 example chunks:
content: Credit Card Bill Payments & Timely Management

Why is timely payment of a credit card bill necessary?
Timely payment of credit card bills is crucial for maintaining financial health and avoiding unnec
metadata : {'source': '..\\data\\text_files\\Credit_card_bill_payment.txt', 'source_file': 'Credit_card_bill_payment.txt', 'file_type': 'text file'}
[Document(metadata={'source': '..\\data\\text_files\\Credit_card_bill_payment.txt', 'source_file': 'Credit_card_bill_payment.txt', 'file_type': 'text file'}, page_content="Credit Card Bill Payments & Timely Management\n\nWhy is timely payment of a credit card bill necessary?\nTimely payment of credit card bills is crucial for maintaining financial health and avoiding unnecessary fees and penalties. Paying your credit card bill on time each month demonstrates responsible financial behavior, which positively impacts credit scores. Late payments result in hefty late fees and interest ch

In [ ]:
#EMBEDDING

import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
from faiss.config import Setting 
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

import os
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS


embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print("Generating embeddings and building FAISS index...")

vector_store = FAISS.from_documents(chunks, embeddings)

DB_FAISS_PATH = "vectorstore/db_faiss"
vector_store.save_local(DB_FAISS_PATH)

print(f"Successfully saved FAISS database to {DB_FAISS_PATH}")

In [10]:
import os
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_community.vectorstores import FAISS

embedding = HuggingFaceBgeEmbeddings(model_name = "all-MiniLM-L6-v2")
print("Genrating embedding and building faiss index")
vector_store = FAISS.from_documents(chunks, embedding)
DB_FAISS_PATH= "vectorstore/db_faiss"
vector_store.save_local(DB_FAISS_PATH)
print(f"succesfully created FAISS DB to {DB_FAISS_PATH}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5811.00it/s]


Genrating embedding and building faiss index
succesfully created FAISS DB to vectorstore/db_faiss


In [ ]:
# Testing
query = "What is the penalty if my outstanding balance is 15,000?"
docs = vector_store.similarity_search(query, k=2) # k=2 returns the top 2 matches

print(f"Found answer in: {docs[0].metadata['source_file']}")
print(docs[0].page_content)

Found answer in: SBI_MITC.txt
1. SBI Late Payment Charges
Outstanding Balance ₹0 to ₹100: Nil penalty.
Outstanding Balance ₹100 to ₹500: ₹100 penalty.
Outstanding Balance ₹501 to ₹1,000: ₹500 penalty.
Outstanding Balance ₹1,001 to ₹10,000: ₹750 penalty.
Outstanding Balance ₹10,001 to ₹25,000: ₹950 penalty.
Outstanding Balance ₹25,001 to ₹50,000: ₹1,100 penalty.
Outstanding Balance greater than ₹50,000: ₹1,300 penalty.
An additional Late Payment Charge of ₹100 will be levied on missing the payment of the Minimum Amount Due (MAD) by the due date for two consecutive cycles.
